In [ ]:
import torch
import torch.nn.functional as F
import requests
import random
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import tqdm
from tqdm.auto import tqdm

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
print(text.__len__())

print(text[:500])

In [ ]:
if (device := torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")):
    print(f"Using device: {device}")

block_size = 16
epochs = 600
lr = 0.001

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print(chars)
def stoi(x):
    return chars.index(x) if isinstance(x, str) else chars[x]
_stoi = {c: i for i, c in enumerate(chars)}
def itos(x):
    return _stoi[x] if isinstance(x, str) else chars[x]

print("itos:", itos(1), itos(2), itos(3), itos(0), "\n")
print("stoi:", stoi('.'), stoi('a'), stoi('b'), stoi('c'), "/n")

In [ ]:
def build_dataset(words):
    X, Y = [], []
    context = [0] * block_size
    for ch in words:
        ix = stoi(ch)
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [ ]:
X, Y = build_dataset(text)
n = X.shape[0]
Xtr, Ytr = X[:int(n*0.9)], Y[:int(n*0.9)]
Xdev, Ydev = X[int(n*0.9):], Y[int(n*0.9):]
Xtr, Ytr = [x.to(device) for x in (Xtr, Ytr)]
Xdev, Ydev = [x.to(device) for x in (Xdev, Ydev)]
print(X.shape, Y.shape)
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)

In [ ]:
trainds = torch.utils.data.TensorDataset(Xtr.cpu(), Ytr.cpu())
traindl = DataLoader(
    dataset=trainds,
    batch_size=8192,
    pin_memory=False,
    shuffle=True,
    num_workers=16,
    prefetch_factor=16
)

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = torch.nn.Embedding(65,32)
        self.net = torch.nn.Sequential(
            nn.Flatten(),
            nn.Linear(512,400,False),
            nn.BatchNorm1d(400),
            nn.Tanh(),
            nn.Linear(400,65)
        )
    def forward(self, Xb):
        return self.net(self.embed(Xb))

        
        

In [ ]:
model = MLP().to(device)
# model = torch.compile(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma = 0.99995)

In [ ]:
model.train()
pbar = tqdm(range(epochs), desc="train")
for e in pbar:
    for Xb, Yb in traindl:
        Xb, Yb = Xb.to(device), Yb.to(device)
        logits = model(Xb)
        loss = F.cross_entropy(logits, Yb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
    pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
    # if e % 10 == 0:
    #     print(e, loss.item(), scheduler.get_last_lr()[0])

In [ ]:
model.eval()
with torch.no_grad():
    ix = torch.randint(0, Xdev.shape[0], (Xdev.shape[0],), device=device)
    Xb, Yb = Xdev[ix], Ydev[ix]
    logits = model(Xb)
    loss = F.cross_entropy(logits, Yb)
    print(loss.item())

In [ ]:
model.eval()
for i in range(10):
    out = []
    context = [0] * block_size
    while True:
        logits = model(torch.tensor([context], device=device))
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(itos(ix))
        if ix == 0:
            break
    print(''.join(out))